<a href="https://colab.research.google.com/github/mandevautospa/GlobalTerror_NeuralNetwork/blob/main/gtd_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import pandas as pd
import kagglehub
import torch
import torch.nn as nn
import requests
import os

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics.pairwise import haversine_distances

path = kagglehub.dataset_download("START-UMD/gtd")
print(tf.version) #ensure version 2.x

Using Colab cache for faster access to the 'gtd' dataset.
<module 'tensorflow._api.v2.version' from '/usr/local/lib/python3.12/dist-packages/tensorflow/_api/v2/version/__init__.py'>


In [ ]:
for f in os.listdir(path):
  if f.endswith(".csv"):
    csv_path = os.path.join(path, f)
df = pd.read_csv(csv_path, encoding ="latin1", low_memory=False)
df = df.replace({-99:None})
df.head()

features = ["iyear", "imonth", "iday",
            "country", "region",
            "latitude", "longitude",
            "nkill", "nwound", "gname"]

targets = ["attacktype1", "success", "gname", "nkill", "nwound"]
all_cols = list(dict.fromkeys(features + targets))
df = df[all_cols].dropna()

encoders = {}
cat_cols = ["country", "region", "gname", "attacktype1"]

for col in cat_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col].astype(str))
  encoders[col] = le

# All numeric feature columns (scaling deferred to after train/val split)
num_cols = ["iyear", "imonth", "iday", "latitude", "longitude", "nkill", "nwound"]


In [ ]:
import os

# iso3_map: map ACLED country names to ISO 3166-1 alpha-3 codes
iso3_map = {}

ACLED_URL = "https://api.acleddata.com/acled/read"

# Retrieve ACLED credentials from Colab userdata or environment variables
try:
    from google.colab import userdata
    acled_email = userdata.get("ACLED_EMAIL")
    acled_key   = userdata.get("ACLED_KEY")
except Exception:
    acled_email = os.environ.get("ACLED_EMAIL", "")
    acled_key   = os.environ.get("ACLED_KEY", "")

params = {"limit": 1000, "year": 2026, "email": acled_email, "key": acled_key}

try:
    r = requests.get(ACLED_URL, params=params, timeout=30)
    r.raise_for_status()
    acled = pd.DataFrame(r.json()['data'])
    acled['event_date_utc'] = pd.to_datetime(acled['event_date'], utc=True)
    acled['country_iso3'] = acled['country'].map(iso3_map)
    acled['reported_by_acled'] = 1
except Exception as e:
    print(f"ACLED data unavailable: {e}")
    acled = pd.DataFrame(columns=['event_date', 'event_date_utc', 'country', 'country_iso3', 'reported_by_acled'])

In [ ]:
print(df.columns.tolist())

['iyear', 'imonth', 'iday', 'country', 'region', 'latitude', 'longitude', 'nkill', 'nwound', 'gname', 'attacktype1', 'success']


In [ ]:
# Temporal train/validation split: train on older data, validate on newer data
# Scaler is fit on training data only to prevent data leakage
# split_year can be adjusted: earlier years give more training data,
# later years give a larger/more-recent validation set
split_year = 2015
df_train = df[df["iyear"] <= split_year].copy()
df_val   = df[df["iyear"] > split_year].copy()

scaler = StandardScaler()
df_train[num_cols] = scaler.fit_transform(df_train[num_cols])
df_val[num_cols]   = scaler.transform(df_val[num_cols])


In [ ]:
X_train = torch.tensor(df_train.drop(columns=targets).values, dtype=torch.float32)
X_val   = torch.tensor(df_val.drop(columns=targets).values, dtype=torch.float32)

y_attacktype_train = torch.tensor(df_train["attacktype1"].values, dtype=torch.long)
y_success_train    = torch.tensor(df_train["success"].values, dtype=torch.float32)
y_group_train      = torch.tensor(df_train["gname"].values, dtype=torch.long)
y_casualties_train = torch.tensor(df_train[["nkill", "nwound"]].values, dtype=torch.float32)

y_attacktype_val   = torch.tensor(df_val["attacktype1"].values, dtype=torch.long)
y_success_val      = torch.tensor(df_val["success"].values, dtype=torch.float32)
y_group_val        = torch.tensor(df_val["gname"].values, dtype=torch.long)
y_casualties_val   = torch.tensor(df_val[["nkill", "nwound"]].values, dtype=torch.float32)


In [ ]:
train_dataset = TensorDataset(X_train, y_attacktype_train)
val_dataset   = TensorDataset(X_val,   y_attacktype_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False)


In [ ]:
print(df.dtypes)


iyear          float64
imonth         float64
iday           float64
country          int64
region           int64
latitude       float64
longitude      float64
nkill          float64
nwound         float64
gname            int64
attacktype1      int64
success          int64
dtype: object
